In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/filtered_complaints.csv"
)

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,91
1,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,156
2,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,233
3,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,454
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem when making payments,Problem during payment process,"Although the account had been deemed closed, I...",Company believes it acted appropriately as aut...,Atlanticus Services Corporation,NY,11212,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13965746,170


In [2]:
df.shape

(191, 19)

In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/complaints.csv",
    usecols=["Product", "Consumer complaint narrative"],
    nrows=10000
)

df.shape

(10000, 2)

In [4]:
df["Product"].value_counts()

Product
Credit reporting or other personal consumer reports        9608
Debt collection                                             290
Credit card                                                  42
Checking or savings account                                  20
Money transfer, virtual currency, or money service           20
Payday loan, title loan, personal loan, or advance loan       7
Vehicle loan or lease                                         6
Mortgage                                                      3
Debt or credit management                                     2
Prepaid card                                                  1
Student loan                                                  1
Name: count, dtype: int64

In [5]:
df = df.dropna(
    subset=["Consumer complaint narrative"]
)

df.shape

(2, 2)

In [6]:
df.shape

(2, 2)

In [7]:
import os

print(os.getcwd())

/home/dag-dagne/rag-complaint-chatbot/notebooks


In [8]:
df = pd.read_csv(
    "../data/raw/complaints.csv",
    usecols=[
        "Product",
        "Consumer complaint narrative"
    ],
    nrows=50000
)

df = df.dropna(
    subset=["Consumer complaint narrative"]
)

print(df.shape)

(672, 2)


In [9]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/complaints.csv",
    usecols=[
        "Product",
        "Consumer complaint narrative"
    ],
    nrows=200000
)

df = df.dropna(
    subset=["Consumer complaint narrative"]
)

print(df.shape)

(4751, 2)


In [10]:
sample_df = df.sample(
    n=len(df),
    random_state=42
)

sample_df.shape

(4751, 2)

In [11]:
sample_df.to_csv(
    "../data/processed/task2_sample.csv",
    index=False
)

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = []

for text in sample_df["Consumer complaint narrative"]:
    chunks = splitter.split_text(text)
    documents.extend(chunks)

print(len(documents))

16273


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
embeddings = model.encode(
    documents,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/509 [00:00<?, ?it/s]

(16273, 384)


In [16]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)

16273


In [ ]:
import faiss
import numpy as np
from pathlib import Path

# create directory if it doesn't exist
Path("../data/vectorstore").mkdir(parents=True, exist_ok=True)

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

faiss.write_index(
    index,
    "../vectorstore/complaints.index"
)

print("Vectors stored:", index.ntotal)

Vectors stored: 16273


In [18]:
len(documents)

16273

In [20]:
import pickle
from pathlib import Path

Path("../vectorstore").mkdir(
    parents=True,
    exist_ok=True
)

with open("../vectorstore/documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("Saved!")

Saved!
